## 1.数据切分 & 基线模型

In [ ]:
import pandas as pd
import numpy as np
import os
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

## 1.数据进行随机切分
def simple_split_by_id(df, id_col='ID', train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=1):
    # 检查是否已存在partition列
    if 'partition' in df.columns:
        print(f"⚠️ 发现已存在的'partition'列，将被覆盖")
        df = df.drop(columns=['partition'])
            
    # 获取唯一ID
    unique_ids = df[id_col].unique()
    
    # 设置随机种子
    np.random.seed(random_seed)
    
    # 打乱ID顺序
    shuffled_ids = np.random.permutation(unique_ids)
    
    # 计算切分点
    n_ids = len(shuffled_ids)
    train_end = int(n_ids * train_ratio)
    val_end = train_end + int(n_ids * val_ratio)
    
    # 分配ID到各个集
    train_ids = shuffled_ids[:train_end]
    val_ids = shuffled_ids[train_end:val_end]
    test_ids = shuffled_ids[val_end:]
    
    # 创建映射
    id_to_partition = {}
    for id_ in train_ids:
        id_to_partition[id_] = 'train'
    for id_ in val_ids:
        id_to_partition[id_] = 'val'
    for id_ in test_ids:
        id_to_partition[id_] = 'test'
    
    # 添加partition列
    df_result = df.copy()
    df_result['partition'] = df_result[id_col].map(id_to_partition)
    
    # 打印统计
    print(f"总ID数: {n_ids}")
    print(f"训练集: {len(train_ids)} IDs ({len(train_ids)/n_ids*100:.1f}%)")
    print(f"验证集: {len(val_ids)} IDs ({len(val_ids)/n_ids*100:.1f}%)")
    print(f"测试集: {len(test_ids)} IDs ({len(test_ids)/n_ids*100:.1f}%)")
    print(f"\n总记录数: {len(df)}")
    for partition in ['train', 'val', 'test']:
        count = len(df_result[df_result['partition'] == partition])
        print(f"{partition}集记录数: {count} ({count/len(df)*100:.1f}%)")
    
    return df_result

# 2. 定义基线模型
models = {
    "RandomForest": (RandomForestRegressor(n_jobs=-1, random_state=42), 
                     {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 20, 30]}),
    "HistGradient": (HistGradientBoostingRegressor(random_state=42), 
                     {'learning_rate': [0.01, 0.1], 'max_iter': [100, 200]})
}

try:
    from xgboost import XGBRegressor
    models['XGBoost'] = (XGBRegressor(n_jobs=-1, random_state=42, objective='reg:absoluteerror'),
                               {'n_estimators': [100, 300, 500], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5, 7]})
    print("-> 已加载 XGBoost")
except ImportError:
    print("-> 未安装 xgboost，跳过")

try:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = (LGBMRegressor(n_jobs=-1, random_state=42, verbose=-1),
                                {'n_estimators': [100, 300, 500], 'learning_rate': [0.01, 0.05, 0.1], 'num_leaves': [31, 63, 127]})
    print("-> 已加载 LightGBM")
except ImportError:
    print("-> 未安装 lightgbm，跳过")

try:
    from sklearn.neural_network import MLPRegressor
    models['MLP_NeuralNet'] = (Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(random_state=42, max_iter=500))]),
                                     {'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)], 'mlp__alpha': [0.0001, 0.001]})
    print("-> 已加载 MLP NeuralNet")

except ImportError:
    pass

print(f"\n当前参与对比的模型列表: {list(models.keys())}")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import BaseCrossValidator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, BayesianRidge, RidgeCV, LassoCV, ElasticNetCV
import joblib
from sklearn.model_selection import KFold, GroupKFold
from sklearn.pipeline import Pipeline
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

output_dir = './model_results'

def bland_altman_analysis(y_true, y_pred, confidence=0.95):
    # 1. 计算差值 (Difference)
    differences = y_true - y_pred
    
    # 2. 计算差值的均值 (Mean difference) - 即偏差
    mean_diff = np.mean(differences)
    
    # 3. 计算差值的标准差 (SD of differences)
    std_diff = np.std(differences, ddof=1)
    
    # 4. 计算95%一致性界限 (95% Limits of Agreement)
    z_score = stats.norm.ppf(1 - (1 - confidence) / 2)  # 95% CI 对应 1.96
    loa_upper = mean_diff + z_score * std_diff
    loa_lower = mean_diff - z_score * std_diff
    
    return loa_lower, loa_upper


feature_group = ['Age', 'Sex', 'Sphere', 'Cylinder', 'Axis', 'AL', 'K1', 'K2', 'ACD']
feature_group_1 = ['Age', 'Sex', 'Sphere', 'Cylinder', 'Axis', 'AL', 'CR', 'ACD','AL/CR', 'ACD/AL', 'AL×ACD', 'AL×CR', 'AL×age']


# 1. 定义特征组合
feature_sets = {
    "group0": feature_group,
    "group1": feature_group_1,
}
# 目标
targets = ['SER']

# 忽略一些搜索过程中的警告，保持输出整洁
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

df_700_final = pd.read_csv('./data/data_700_example.csv')
df_500_final = pd.read_csv('./data/data_500_example.csv')
df_ningbo_final = pd.read_csv('./data/data_ningbo_example.csv')

seeds = list(range(1,5))
for seed in seeds:
    print(f'=========开始在随机seed:{seed} 下切分数据，进行训练.============')
    df_700_final_partition = simple_split_by_id(df_700_final, random_seed = seed)
    
    df_train = df_700_final_partition[df_700_final_partition['partition'] == 'train'].copy()
    df_val = df_700_final_partition[df_700_final_partition['partition'] == 'val'].copy()
    df_inner_test = df_700_final_partition[df_700_final_partition['partition'] == 'test'].copy()
    
    df_500_test = df_500_final.copy()
    df_ningbo_test = df_ningbo_final.copy()
    
    if len(df_train) == 0 or len(df_val) == 0:
        raise ValueError("错误：训练集或验证集为空，请检查 CSV 中的 partition 列内容！")
    
    X_search = pd.concat([df_train, df_val])
    y_search = X_search[targets]
    
    split_indices = np.full(len(X_search), -1)
    split_indices[len(df_train):] = 0
    ps = PredefinedSplit(split_indices)
    
    print(f"\n开始在 {len(feature_sets)} 个特征集 x {len(models)} 个模型上进行训练...")

    results = []
    best_models = {} # 初始化字典存储最佳模型
    flag = False
    
    #每组特征单独进行训练
    for fs_name, features in feature_sets.items():
        print(f"\n===== 特征集: {fs_name} =====")
        
        # 检查特征是否存在，防止报错
        missing_cols = [c for c in features if c not in X_search.columns]
        if missing_cols:
            print(f"  [警告] 缺少列 {missing_cols}，跳过该特征集")
            continue

        try:
            target = 'M_post'
            estimators = []
            best_inner_mae, best_inner_mse,best_inner_r2  = 100.0,100.0,0.0
            best_500_mae, best_500_mse,best_500_r2  = 100.0,100.0,0.0
            best_ningbo_mae, best_ningbo_mse,best_ningbo_r2  = 100.0,100.0,0.0
            
            #单模型训练
            for model_name, (model_obj, params) in models.items():
                print(f"  -> 正在训练 {model_name}...", end="")
                 
                # 使用预定义的验证集 (ps) 进行随机搜索
                search = RandomizedSearchCV(model_obj, params, cv=ps, scoring='neg_mean_absolute_error', 
                                            n_iter=50, random_state=42, n_jobs=-1, refit=False)
                search.fit(X_search[features], y_search[target])
                
                # 获取最佳模型并在独立的 Test 集上评估
                val_scores = search.cv_results_['mean_test_score']  # 每个参数组合在验证集上的平均分
                best_idx = np.argmax(val_scores)
                best_params = search.cv_results_['params'][best_idx]
                best_m = model_obj.set_params(**best_params)
                
                y_test_pred_inner = best_m.predict(df_inner_test[features])
                y_test_pred_500 = best_m.predict(df_500_test[features])
                y_test_pred_ningbo = best_m.predict(df_ningbo_test[features])
                
                mae_test_inner, mae_test_500, mae_test_ningbo = mean_absolute_error(df_inner_test[target], y_test_pred_inner), mean_absolute_error(df_500_test[target], y_test_pred_500),mean_absolute_error(df_ningbo_test[target], y_test_pred_ningbo) 
                mse_test_inner, mse_test_500, mse_test_ningbo = mean_squared_error(df_inner_test[target], y_test_pred_inner), mean_squared_error(df_500_test[target], y_test_pred_500), mean_squared_error(df_ningbo_test[target], y_test_pred_ningbo)
                r2_test_inner, r2_test_500, r2_test_ningbo = r2_score(df_inner_test[target], y_test_pred_inner), r2_score(df_500_test[target], y_test_pred_500), r2_score(df_ningbo_test[target], y_test_pred_ningbo)
                loa_lower_inner, loa_upper_inner = bland_altman_analysis(df_inner_test[target], y_test_pred_inner)
                loa_lower_500, loa_upper_500 = bland_altman_analysis(df_500_test[target], y_test_pred_500)
                loa_lower_ningbo, loa_upper_ningbo = bland_altman_analysis(df_ningbo_test[target], y_test_pred_ningbo)
                
                if best_inner_mae > mae_test_inner:
                    best_inner_mae = mae_test_inner
                if best_inner_mse > mse_test_inner:
                    best_inner_mse = mse_test_inner
                if best_inner_r2 < r2_test_inner:
                    best_inner_r2 = r2_test_inner

                if best_500_mae > mae_test_500:
                    best_500_mae = mae_test_500
                if best_500_mse > mse_test_500:
                    best_500_mse = mse_test_500
                if best_500_r2 < r2_test_500:
                    best_500_r2 = r2_test_500

                if best_ningbo_mae > mae_test_ningbo:
                    best_ningbo_mae = mae_test_ningbo
                if best_ningbo_mse > mse_test_ningbo:
                    best_ningbo_mse = mse_test_ningbo
                if best_ningbo_r2 < r2_test_ningbo:
                    best_ningbo_r2 = r2_test_ningbo
                    
                # 记录详细结果
                results.append({
                    'Feature_Set': fs_name, 
                    'Model': model_name, 
                    'Target': target,
                    'Val_MAE': -search.best_score_, # 验证集最佳分数
                    'Test_inner_MAE': mae_test_inner,           # 测试集实际分数
                    'Test_500_MAE': mae_test_500,
                    'Test_ningbo_MAE': mae_test_ningbo,
                    'Test_inner_MSE': mse_test_inner,           # 测试集实际分数
                    'Test_inner_RMSE': np.sqrt(mse_test_inner),
                    'Test_500_MSE': mse_test_500,
                    'Test_500_RMSE': np.sqrt(mse_test_500),
                    'Test_ningbo_MSE': mse_test_ningbo,
                    'Test_ningbo_RMSE': np.sqrt(mse_test_ningbo),
                    'Test_inner_R2': r2_test_inner,           # 测试集实际分数
                    'Test_500_R2': r2_test_500,
                    'Test_ningbo_R2': r2_test_ningbo,
                    'Test_inner_loa': (loa_lower_inner, loa_upper_inner),
                    'Test_500_loa': (loa_lower_500, loa_upper_500),
                    'Test_ningbo_loa': (loa_lower_ningbo, loa_upper_ningbo),
                    'best_params': search.best_params_
                })
                
                # 保存模型对象
                model_key = f"{fs_name}_{model_name}_{target}"
                best_models[model_key] = best_m
                estimators.append((f"{model_name}_{fs_name}", best_m))
                
                print(f"特征集: {fs_name} 模型 {model_name} 训练完成。")

            #所有模型参与集成
            print("正在训练元模型 (Meta-Model)...")
            stack_reg = StackingRegressor(
                estimators=estimators,
                final_estimator = Ridge(alpha=10.0, random_state=42),
                cv=5, 
                n_jobs=-1,
                passthrough=False # 元模型只看基模型的预测值，不看原始特征
            )
            
            # 重新 Fit (StackingRegressor 会自动对基模型做 CV 预测来训练元模型)
            # 这需要一点时间
            stack_reg.fit(X_search[features], y_search[target])
            
            # 4. 最终评估
            y_pred_stack_inner = stack_reg.predict(df_inner_test[features])
            y_pred_stack_500 = stack_reg.predict(df_500_test[features])
            y_pred_stack_ningbo = stack_reg.predict(df_ningbo_test[features])
            
            mae_stack_inner, mae_stack_500, mae_stack_ningbo  = mean_absolute_error(df_inner_test[target], y_pred_stack_inner), mean_absolute_error(df_500_test[target], y_pred_stack_500), mean_absolute_error(df_ningbo_test[target], y_pred_stack_ningbo)
            mse_stack_inner, mse_stack_500, mse_stack_ningbo  = mean_squared_error(df_inner_test[target], y_pred_stack_inner), mean_squared_error(df_500_test[target], y_pred_stack_500), mean_squared_error(df_ningbo_test[target], y_pred_stack_ningbo)
            r2_stack_inner, r2_stack_500, r2_stack_ningbo  = r2_score(df_inner_test[target], y_pred_stack_inner), r2_score(df_500_test[target], y_pred_stack_500), r2_score(df_ningbo_test[target], y_pred_stack_ningbo)

            loa_lower_stack_inner, loa_upper_stack_inner = bland_altman_analysis(df_inner_test[target], y_pred_stack_inner)
            loa_lower_stack_500, loa_upper_stack_500 = bland_altman_analysis(df_500_test[target], y_pred_stack_500)
            loa_lower_stack_ningbo, loa_upper_stack_ningbo = bland_altman_analysis(df_ningbo_test[target], y_pred_stack_ningbo)
            
            print(f"\n[最终结果] Stacking 集成模型.")
            print(f"MAE\t\t{mae_stack_inner:.6f}\t{mae_stack_500:.6f}\t{mae_stack_ningbo:.6f}")
            print(f"MSE\t\t{mse_stack_inner:.6f}\t{mse_stack_500:.6f}\t{mse_stack_ningbo:.6f}")
            print(f"R2\t\t{r2_stack_inner:.6f}\t{r2_stack_500:.6f}\t{r2_stack_ningbo:.6f}")
            print(f"对比 内部测试集合 Top 1 单模型 inner: {best_inner_mae:.6f}\t{best_inner_mse:.6f}\t{best_inner_r2:.6f}")

            # 记录stacking模型详细结果
            results.append({
                'Feature_Set': fs_name, 
                'Model': 'Stacking', 
                'Target': target,
                'Val_MAE': 0.0, # 验证集最佳分数
                'Test_inner_MAE': mae_stack_inner,           # 测试集实际分数
                'Test_500_MAE': mae_stack_500,
                'Test_ningbo_MAE': mae_stack_ningbo,
                'Test_inner_MSE': mse_stack_inner,           # 测试集实际分数
                'Test_inner_RMSE': np.sqrt(mse_stack_inner),
                'Test_500_MSE': mse_stack_500,
                'Test_500_RMSE': np.sqrt(mse_stack_500),
                'Test_ningbo_MSE': mse_stack_ningbo,
                'Test_ningbo_RMSE': np.sqrt(mse_stack_ningbo),
                'Test_inner_R2': r2_stack_inner,           # 测试集实际分数
                'Test_500_R2': r2_stack_500,
                'Test_ningbo_R2': r2_stack_ningbo,
                'Test_inner_loa': (loa_lower_stack_inner, loa_upper_stack_inner),
                'Test_500_loa': (loa_lower_stack_500, loa_upper_stack_500),
                'Test_ningbo_loa': (loa_lower_stack_ningbo, loa_upper_stack_ningbo),
                'best_params': '-'
            })
    
            # 保存模型对象
            model_key = f"{fs_name}_Stacking_M_post"
            best_models[model_key] = stack_reg
                
        except Exception as e:
            print(f"\n  特征集: {fs_name}  训练失败: {e}")

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df.to_csv(os.path.join(output_dir, 'model_comparison_results_' + str(seed) +'.csv'), index=False, encoding='utf-8-sig')
        print(f"\nseed:{seed}, 训练结束！结果已保存.")
    else:
        print("\n训练未生成任何结果，请检查数据或模型配置。")